In [1]:
# 1. Install & Imports
!pip install -q transformers datasets accelerate torch torchvision scikit-learn

import numpy as np
import random
import os
import json
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import RandAugment
from datasets import load_dataset
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from google.colab import files

In [2]:
# --- 2. KONFIGURASI FINAL ---
OUTPUT_DIR = "./vit_final_fixed"
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 3e-5
SEED = 24

# FITUR AUGMENTASI
USE_RAND_AUGMENT = True
USE_MIXUP_CUTMIX = True
MIX_PROB = 0.5
MIXUP_ALPHA = 0.8
CUTMIX_ALPHA = 1.0

# FITUR REGULARISASI
LABEL_SMOOTHING = 0.1
DROPOUT = 0.1
WEIGHT_DECAY = 0.03

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

In [3]:
# --- 3. Setup Kaggle & Dataset ---
if not os.path.exists('kaggle.json'):
    print("\n📂 Silakan upload file kaggle.json:")
    files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

if not os.path.exists('dataset_binary'):
    print("\n⬇️ Mendownload dataset...")
    !kaggle datasets download -d fanconic/skin-cancer-malignant-vs-benign --force
    !unzip -q -o skin-cancer-malignant-vs-benign.zip -d dataset_binary
    print("✅ Dataset berhasil diekstrak!")

DATA_DIR = "/content/dataset_binary"


📂 Silakan upload file kaggle.json:


Saving kaggle.json to kaggle.json

⬇️ Mendownload dataset...
Dataset URL: https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign
License(s): unknown
 96% 313M/325M [00:01<00:00, 263MB/s]
100% 325M/325M [00:01<00:00, 316MB/s]
✅ Dataset berhasil diekstrak!


In [4]:
# --- 4. PREPROCESSING & PEMBAGIAN DATASET (DIUBAH DI SINI) ---
print("\n⚙️ Menyiapkan Transformasi...")
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

def preprocess_train(examples):
    examples["pixel_values"] = [train_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

def preprocess_val(examples):
    examples["pixel_values"] = [val_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples


⚙️ Menyiapkan Transformasi...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [5]:
# MENGATASI DATA LEAKAGE: Memecah data latih
kaggle_train = load_dataset("imagefolder", data_dir=os.path.join(DATA_DIR, "train"), split="train")
test_ds  = load_dataset("imagefolder", data_dir=os.path.join(DATA_DIR, "test"), split="train")

train_val_split = kaggle_train.train_test_split(test_size=0.2, seed=SEED)
train_ds = train_val_split['train']
val_ds   = train_val_split['test']

print(f"Jumlah Data Latih (Training)     : {len(train_ds)}")
print(f"Jumlah Data Validasi (Validation): {len(val_ds)}")
print(f"Jumlah Data Uji (Test)           : {len(test_ds)}")

Resolving data files:   0%|          | 0/2637 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Jumlah Data Latih (Training)     : 2109
Jumlah Data Validasi (Validation): 528
Jumlah Data Uji (Test)           : 660


In [6]:
# --- 5. Hitung Class Weights (dari 80% data latih) ---
try:
    labels = train_ds['label']
    labels = np.array(labels)
except:
    labels = [item['label'] for item in train_ds]
    labels = np.array(labels)

class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"✅ Class Weights: {class_weights.tolist()}")

✅ Class Weights: [0.9185540080070496, 1.0972944498062134]


In [7]:
# Terapkan Transformasi
train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_val)      # Menerapkan transform ke data validasi
test_ds.set_transform(preprocess_val)

In [8]:
# --- 6. ENGINE MIXUP & CUTMIX ---
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def mixup_cutmix_collator(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    num_classes = 2
    one_hot_labels = F.one_hot(labels, num_classes=num_classes).float()

    if USE_MIXUP_CUTMIX and np.random.rand() < MIX_PROB:
        lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        rand_index = torch.randperm(pixel_values.size(0))

        target_a = one_hot_labels
        target_b = one_hot_labels[rand_index]

        if np.random.rand() < 0.5:
            # CUTMIX
            bbx1, bby1, bbx2, bby2 = rand_bbox(pixel_values.size(), lam)
            pixel_values[:, :, bby1:bby2, bbx1:bbx2] = pixel_values[rand_index, :, bby1:bby2, bbx1:bbx2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (pixel_values.size(-1) * pixel_values.size(-2)))
        else:
            # MIXUP
            pixel_values = lam * pixel_values + (1 - lam) * pixel_values[rand_index]

        mixed_labels = lam * target_a + (1 - lam) * target_b
        return {"pixel_values": pixel_values, "labels": mixed_labels}

    return {"pixel_values": pixel_values, "labels": one_hot_labels}

In [9]:
# --- 7. MODEL SETUP ---
print("🧠 Membangun Model ViT dengan Dropout...")
model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "benign", 1: "malignant"},
    label2id={"benign": 0, "malignant": 1},
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT
)

🧠 Membangun Model ViT dengan Dropout...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
# --- 8. LOSS FUNCTION & METRICS ---
class FinalLoss(nn.Module):
    def __init__(self, class_weights=None, label_smoothing=0.0):
        super().__init__()
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        device = logits.device
        w = self.class_weights.to(device) if self.class_weights is not None else None
        targets = targets.to(device)
        loss = F.cross_entropy(logits, targets, weight=w, label_smoothing=self.label_smoothing)
        return loss

final_loss_fn = FinalLoss(class_weights=class_weights, label_smoothing=LABEL_SMOOTHING)

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    if len(labels.shape) > 1:
        labels = np.argmax(labels, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

In [12]:
# --- 9. TRAINER SETUP ---
class FinalTrainer(Trainer):
    def __init__(self, custom_loss_fn=None, **kwargs):
        super().__init__(**kwargs)
        self.custom_loss_fn = custom_loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = self.custom_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_pin_memory=True,
    logging_steps=50,
    remove_unused_columns=False
)

trainer = FinalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,           # <--- MENGGUNAKAN DATA VALIDASI BUKAN DATA UJI
    processing_class=processor,
    data_collator=mixup_cutmix_collator,
    compute_metrics=compute_metrics,
    custom_loss_fn=final_loss_fn
)

In [13]:
# --- 10. EKSEKUSI TRAINING ---
print("\n🚀 Memulai Training Final (Proposed) + Mixup/Cutmix...")
trainer.train()


🚀 Memulai Training Final (Proposed) + Mixup/Cutmix...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.554835,0.481336,0.831439,0.831910
2,0.499297,0.427014,0.890152,0.890056
3,0.464814,0.459120,0.867424,0.867245
4,0.483978,0.413106,0.890152,0.889887
5,0.461363,0.527838,0.842803,0.838997
6,0.424052,0.383264,0.895833,0.896082
7,0.433641,0.423029,0.897727,0.897869
8,0.425365,0.444349,0.857955,0.858343
9,0.409562,0.449150,0.865530,0.865886
10,0.411698,0.371952,0.901515,0.901163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6600, training_loss=0.34530294331637296, metrics={'train_runtime': 7124.2855, 'train_samples_per_second': 29.603, 'train_steps_per_second': 0.926, 'total_flos': 1.6343060609717453e+19, 'train_loss': 0.34530294331637296, 'epoch': 100.0})

In [14]:
# --- 11. EVALUASI AKHIR PADA DATA TEST ---
print("\n📊 Melakukan Evaluasi Akhir pada Data Test Murni...")

# Collator tanpa augmentasi untuk pengujian
def clean_collator(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    num_classes = 2
    one_hot_labels = F.one_hot(labels, num_classes=num_classes).float()
    return {"pixel_values": pixel_values, "labels": one_hot_labels}

trainer.data_collator = clean_collator

# PREDIKSI DI TEST SET
preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

if len(y_true.shape) > 1:
    y_true = np.argmax(y_true, axis=1)

final_acc = accuracy_score(y_true, y_pred)
final_f1 = f1_score(y_true, y_pred, average='weighted')
cm = confusion_matrix(y_true, y_pred).tolist()

# EKSTRAK LOG HISTORY UNTUK JSON
log_history = trainer.state.log_history
epochs_val, val_loss, val_acc, train_loss = [], [], [], []

for entry in log_history:
    if 'loss' in entry:
        train_loss.append(entry['loss'])
    elif 'eval_loss' in entry:
        epochs_val.append(entry['epoch'])
        val_loss.append(entry['eval_loss'])
        if 'eval_accuracy' in entry:
            val_acc.append(entry['eval_accuracy'])
        elif 'eval_acc' in entry:
            val_acc.append(entry['eval_acc'])

if len(train_loss) > len(val_loss):
    train_loss = train_loss[:len(val_loss)]

# SIMPAN JSON
results = {
    "name": "Skenario Final (RandAugment + Mixup + Cutmix)",
    "epochs": epochs_val,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_acc": val_acc,
    "final_test_accuracy": final_acc,
    "final_test_f1": final_f1,
    "confusion_matrix": cm
}

filename = "metrics_scenario_final_mixup.json"
with open(filename, "w") as f:
    json.dump(results, f)

print(f"\n✅ Training Selesai! Hasil tersimpan di '{filename}'")
print(f"🏆 Final Accuracy: {final_acc:.4f}")
try:
    files.download(filename)
except:
    pass


📊 Melakukan Evaluasi Akhir pada Data Test Murni...



✅ Training Selesai! Hasil tersimpan di 'metrics_scenario_final_mixup.json'
🏆 Final Accuracy: 0.9258


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# SIMPAN DAN DOWNLOAD MODEL
print("\n📦 Mengompres & Download Model untuk Deployment...")
model_dir = "vit_final_model_mixup"
trainer.save_model(model_dir)
processor.save_pretrained(model_dir)

shutil.make_archive("vit_final_proposed_mixup", 'zip', model_dir)
try:
    files.download("vit_final_proposed_mixup.zip")
except:
    pass
print("✅ Selesai! JSON Metrics dan File Model (.zip) sedang didownload.")


📦 Mengompres & Download Model untuk Deployment...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Selesai! JSON Metrics dan File Model (.zip) sedang didownload.
